### PDF Question Answering Retriever

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/home/balaji/Desktop/PDF_QA_retriver/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
### Read all the pdf
def process_all_pdfs(pdf_directory):
    """Process all PDF files in the given directory and split their content into chunks."""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # find all pdf files recursively
    pdf_files = list(pdf_dir.rglob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process.")
    
    
    for pdf_file in pdf_files:
        print(f"Processing file: {pdf_file}")
        try:
            loader= PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["source_path"] = 'pdf'
            
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"Error loading {pdf_file}: {e}") 
            
    print(f"Total documents loaded: {len(all_documents)}")
    return all_documents

# process all the pdfs in the data directory
all_pdf_documents = process_all_pdfs("../data/pdf")


Found 0 PDF files to process.
Total documents loaded: 0


In [ ]:


def load_all_pdfs(data_dir: str):
    """
    Recursively loads all PDF files from a given directory.
    Adds source metadata and prints total documents loaded.
    Returns a list of LangChain Document objects.
    """
    all_documents = []

    # 1️⃣ Find all PDF files recursively
    for root, _, files in os.walk(data_dir):
        for file in files:
            if file.lower().endswith(".pdf"):
                file_path = os.path.join(root, file)
                print(f"📄 Loading: {file_path}")

                # 2️⃣ Load the PDF
                loader = PyPDFLoader(file_path)
                pdf_docs = loader.load()

                # 3️⃣ Add source info in metadata
                for doc in pdf_docs:
                    doc.metadata["source"] = file_path

                # 4️⃣ Append to final list
                all_documents.extend(pdf_docs)

    # 5️⃣ Split text into chunks (recommended for RAG)
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        separators=["\n\n", "\n", ".", " "]
    )
    split_docs = text_splitter.split_documents(all_documents)

    print(f"\n✅ Total documents loaded: {len(split_docs)}")
    return split_docs

all_pdf_documents